In [2]:
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib.pyplot as plt
import mlflow
import json
from pathlib import Path

# ==========================================
# 1. SETUP PATH & MLFLOW
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("09_SHAP_Explainability")

print("🔍 1. Memuat Data Testing dan Model Final XGBoost...")
test_df = pd.read_csv(root_path / "Data/split/test_data.csv")

# 35 Fitur MFO
selected_features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1', 'VCL4', 'VCL5', 'VCL6', 
                     'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'gender', 'engnat', 'age', 'religion', 'orientation',
                     'race', 'voted', 'married', 'familysize']
target_labels = ['Depression', 'Anxiety', 'Stress']

X_test = test_df[selected_features]

# Load Model
model_path = root_path / "models" / "multilabel_xgboost_mfo.pkl"
with open(model_path, "rb") as f:
    multi_target_model = pickle.load(f)

# Folder output untuk gambar SHAP
output_dir = root_path / "outputs" / "shap_figures"
output_dir.mkdir(parents=True, exist_ok=True)

with mlflow.start_run(run_name="SHAP_Analysis"):
    print("🧠 2. Memulai Ekstraksi SHAP Values (Membongkar Otak AI)...")
    
    # Karena kita pakai MultiOutputClassifier, di dalamnya ada 3 model XGBoost terpisah
    # Kita harus melakukan SHAP explainer untuk masing-masing kondisi mental
    
    for i, label in enumerate(target_labels):
        print(f"   -> Menganalisis alasan AI untuk kondisi: {label}...")
        
        # Mengambil model XGBoost spesifik untuk target ke-i
        xgb_estimator = multi_target_model.estimators_[i]

        # Membuat TreeExplainer khusus algoritma berbasis pohon (XGBoost)
        explainer = shap.TreeExplainer(xgb_estimator)
        
        # Menghitung SHAP values (Gunakan sebagian data test agar tidak terlalu lama)
        # Jika laptop kuat, bisa hapus sample(500) menjadi X_test murni
        X_sample = X_test.sample(500, random_state=42) if len(X_test) > 500 else X_test
        shap_values = explainer.shap_values(X_sample)
        
        # ==========================================
        # 3. MENCETAK GRAFIK SHAP SUMMARY PLOT
        # ==========================================
        plt.figure(figsize=(10, 8))
        
        # Membuat beeswarm plot
        shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
        
        plt.title(f'SHAP Feature Importance: Faktor Pemicu {label}', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        
        # Simpan gambar
        fig_path = output_dir / f"shap_summary_{label.lower()}.png"
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        # Log ke MLflow
        mlflow.log_artifact(str(fig_path))

    print("\n" + "="*50)
    print("🎉 TAHAP 09 SELESAI: KOTAK HITAM BERHASIL DIBONGKAR!")
    print(f"Silakan buka folder {output_dir} untuk melihat 3 grafik SHAP.")
    print("="*50)

🔍 1. Memuat Data Testing dan Model Final XGBoost...
🧠 2. Memulai Ekstraksi SHAP Values (Membongkar Otak AI)...
   -> Menganalisis alasan AI untuk kondisi: Depression...
   -> Menganalisis alasan AI untuk kondisi: Anxiety...
   -> Menganalisis alasan AI untuk kondisi: Stress...

🎉 TAHAP 09 SELESAI: KOTAK HITAM BERHASIL DIBONGKAR!
Silakan buka folder d:\Amikom\Semester 6\Project Data Mining\Project\outputs\shap_figures untuk melihat 3 grafik SHAP.
